# 📐 Module 3 — Class 2 Homework: Scaling

**Topic:** Scaling — StandardScaler, MinMaxScaler, RobustScaler

## 📋 What you have to do

This notebook **already has working code**. Your job:

1. **Run every cell from top to bottom** — it should run with no errors.
2. **Add a comment on EVERY code cell** that explains what the cell does, in your own words.
   - Use `#` to write comments inside the code cell.
   - Write at least 1 short sentence per cell. Longer is better.
3. **Save your notebook** as `Module<X>_Class<Y>_<YourName>.ipynb` and submit.

**Example of a good commented cell:**

```python
# Count how many customers churned vs stayed
# value_counts() returns the number of rows for each unique value
df['Churn'].value_counts()
```

**Example of a BAD comment (do not do this):**

```python
# count
df['Churn'].value_counts()
```

---


In [23]:
# === SETUP — run this first ===
import pandas as pd # Data upload and read
import numpy as np # Calculs for data
import matplotlib.pyplot as plt # Data visualize
import warnings; warnings.filterwarnings('ignore') #Blocks all warning messages that appear during program execution and does not show them to the user (ignore).

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv' # Upload data url
df = pd.read_csv(url) # Read data
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0) # TotalCharges column Empty value fillna 0
print('Loaded:', df.shape) # Data shape


Loaded: (7043, 21)


### Cell 1 — see how different the ranges are

In [24]:
cols = ['tenure', 'MonthlyCharges', 'TotalCharges'] # select column
df[cols].describe().round(2) # Value count, mean, std, min,iqr, max rount poin 2 value

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,32.37,64.76,2279.73
std,24.56,30.09,2266.79
min,0.00,18.25,0.00
25%,9.00,35.50,398.55
50%,29.00,70.35,1394.55
75%,55.00,89.85,3786.60
max,72.00,118.75,8684.80


### Cell 2 — apply StandardScaler (mean=0, std=1)

In [25]:
from sklearn.preprocessing import StandardScaler #It imports a class called StandardScaler from the data preprocessing section of the scikit-learn library into the program.
scaler = StandardScaler() #It creates a copy (object) of this class and loads it into a variable called scaler. Now we can standardize our data using this scaler.
X = df[cols]  # Only need cols select
X_std = scaler.fit_transform(X) #Standardizes the extracted data (brings all numbers to the same scale and transfers them to a NumPy array)
X_std_df = pd.DataFrame(X_std, columns=cols) #It returns that standardized array to a beautiful Pandas table (DataFrame) and puts the column names back in place.
X_std_df.describe().round(2) # New data frame count, mean, std, min,iqr, max rount poin 2 value

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,-0.00,-0.00,-0.00
std,1.00,1.00,1.00
min,-1.32,-1.55,-1.01
25%,-0.95,-0.97,-0.83
50%,-0.14,0.19,-0.39
75%,0.92,0.83,0.66
max,1.61,1.79,2.83


### Cell 3 — apply MinMaxScaler (range 0 to 1)

In [26]:
from sklearn.preprocessing import MinMaxScaler #It imports a class called MinMaxScaler from the data preprocessing section of the scikit-learn library into the program.
mm = MinMaxScaler() # Initialize the 0-1 scaler
X_mm = mm.fit_transform(X) # Scale the data between 0 and 1
X_mm_df = pd.DataFrame(X_mm, columns=cols) # Convert the scaled array back to a DataFrame
X_mm_df.agg(['min', 'max']).round(2) # Double-check that all mins are 0 and maxs are 1

,tenure,MonthlyCharges,TotalCharges
min,0.0,0.0,0.0
max,1.0,1.0,1.0


### Cell 4 — apply RobustScaler (uses median + IQR, ignores outliers)

In [27]:
from sklearn.preprocessing import RobustScaler #It imports a class called RobustScaler from the data preprocessing section of the scikit-learn library into the program.
rb = RobustScaler() # Initialize the robust scaler (handles outliers well)
X_rb = rb.fit_transform(X) # Scale features using median and quantiles
X_rb_df = pd.DataFrame(X_rb, columns=cols) # Convert the scaled array back to a DataFrame
X_rb_df.describe().round(2) # View summary statistics to verify the scaling

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,0.07,-0.10,0.26
std,0.53,0.55,0.67
min,-0.63,-0.96,-0.41
25%,-0.43,-0.64,-0.29
50%,0.00,0.00,0.00
75%,0.57,0.36,0.71
max,0.93,0.89,2.15


### Cell 5 — split FIRST, then scale (the big rule — no data leakage)

In [28]:
from sklearn.model_selection import train_test_split # Import the train/test split utility
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42) # Split data into 80% training and 20% testing sets
scaler = StandardScaler() # Initialize the standard scaler
X_train_s = scaler.fit_transform(X_train) # Fit scaler on training data and transform it
X_test_s = scaler.transform(X_test) # Transform test data using the training data's scale (No fitting!)
print(f'Train shape: {X_train_s.shape}, Test shape: {X_test_s.shape}') # Print the dataset dimensions to verify the split
print(f'Train mean (should be ~0): {X_train_s.mean(axis=0).round(2)}') # Verify that the training data mean is now 0

Train shape: (5634, 3), Test shape: (1409, 3)
Train mean (should be ~0): [ 0. -0.  0.]


### Cell 6 — save the trained scaler to disk

In [29]:
import joblib # Import joblib for saving and loading Python objects
joblib.dump(scaler, 'my_scaler.joblib') # Save the trained scaler to a file
loaded = joblib.load('my_scaler.joblib') # Load the saved scaler back into memory
print('Saved and loaded. Learned mean:', loaded.mean_.round(2)) # Verify the loaded scaler has kept its learned parameters

Saved and loaded. Learned mean: [  32.37   64.86 2287.09]
